# Train — DBpedia (in-batch negatives)

**Purpose:** Trains `HybridSPARQLEncoder` on DBpedia semantic triplets using **in-batch negatives** (InfoNCE): one positive per anchor, with the other batch items acting as negatives.

**Input:** `../../data/jaccard_triplets_dbpedia.json` — unzip `data/jaccard_triplets_dbpedia.zip` once before running.

**Splits:** `../../data/splits_dbpedia.json` (seed 42 — train 3770 / valid 471 / test 472). Set `REGENERATE_SPLITS = True` to redo.

**Output:** `../../checkpoints/dbpedia_model_semantic-20.pt` — trained model weights.

**Config:** see `sparqlce.config.Config` (device, epochs, learning rate, temperature). Batch size for in-batch training is 16.

**Note:** As of the canonicalization refactor, `SPARQLGraphBuilder` automatically canonicalizes variables and drops no-op `FILTER(true)`. To additionally train with random variable-rename augmentation, pass `augment_rename_prob=0.3` to `SemanticSPARQLDataset(...)`.

**Sections:** 1. Imports and data load | 2. Splits | 3. Dataset and DataLoaders (in-batch) | 4. Loss | 5. Training and save.


In [1]:
import os
import sys
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# 1. Setup Project Paths & Environment
PROJECT_ROOT = os.path.abspath("../..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# 2. Imports from custom project modules
from sparqlce.config import Config
from sparqlce.graph_builder import SPARQLGraphBuilder
from sparqlce.encoder import HybridSPARQLEncoder
from sparqlce.dataset import SemanticSPARQLDataset, semantic_collate_fn_inbatch

# 3. Reproducibility & Device Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 4. Define In-Batch InfoNCE Loss
def infonce_inbatch(a_emb, p_emb, temperature=0.07):
    a_emb = F.normalize(a_emb, dim=1)
    p_emb = F.normalize(p_emb, dim=1)
    logits = torch.matmul(a_emb, p_emb.T) / temperature
    labels = torch.arange(a_emb.size(0), device=a_emb.device)
    return F.cross_entropy(logits, labels)

# 5. Initialize Graph Builder & Full Dataset 
# Note: in_batch_negatives=True outputs the 3-element pairs expected by your collate function.
graph_builder = SPARQLGraphBuilder() 
dataset = SemanticSPARQLDataset(
    "../../data/jaccard_triplets_dbpedia.json",
    graph_builder=graph_builder,
    in_batch_negatives=True  
)

# 6. Dynamically generate perfectly sized splits (80/10/10) to avoid IndexErrors
total_size = len(dataset)
train_size = int(0.80 * total_size)
val_size = int(0.10 * total_size)
test_size = total_size - train_size - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    dataset, 
    [train_size, val_size, test_size], 
    generator=generator
)

# 7. Create DataLoaders
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, collate_fn=semantic_collate_fn_inbatch
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, collate_fn=semantic_collate_fn_inbatch
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, collate_fn=semantic_collate_fn_inbatch
)

# 8. Initialize Model and Optimizer
model = HybridSPARQLEncoder().to(device) 
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 9. Training Hyperparameters & Directory Setup
EPOCHS = Config.EPOCHS if hasattr(Config, 'EPOCHS') else 10
TEMPERATURE = Config.TEMPERATURE if hasattr(Config, 'TEMPERATURE') else 0.07
LAMBDA_REG = 0.3
best_val_sim = 0.0

checkpoint_dir = "../../checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"✓ Training on {device}")
print(f"✓ Dataset split successfully: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

# 10. Training Loop
for epoch in range(1, EPOCHS + 1):
    # ---- Train ----
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:2d}/{EPOCHS} [Train]")
    
    for batch in train_pbar:
        anchor_graphs = batch[0].to(device)
        pos_graphs = batch[1].to(device)
        pos_js = batch[2].to(device)
        
        optimizer.zero_grad()
        
        a_emb = model(anchor_graphs)
        p_emb = model(pos_graphs)
        
        # InfoNCE loss
        loss_ce = infonce_inbatch(a_emb, p_emb, TEMPERATURE)
        
        # L2 normalize embeddings for stable Jaccard calibration regression
        a_emb_norm = F.normalize(a_emb, dim=1)
        p_emb_norm = F.normalize(p_emb, dim=1)
        cos_sim = (a_emb_norm * p_emb_norm).sum(dim=1)
        
        loss_reg = (cos_sim - pos_js).pow(2).mean()
        loss = loss_ce + LAMBDA_REG * loss_reg
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        train_pbar.set_postfix(
            loss=f"{loss.item():.4f}", 
            ce=f"{loss_ce.item():.4f}", 
            reg=f"{loss_reg.item():.4f}"
        )
    
    train_loss /= len(train_loader)
    
    # ---- Validation ----
    model.eval()
    val_sim = 0.0
    val_count = 0
    
    with torch.no_grad():
        for batch in val_loader:
            anchor_graphs = batch[0].to(device)
            pos_graphs = batch[1].to(device)
            
            a_emb = model(anchor_graphs)
            p_emb = model(pos_graphs)
            
            a_emb = F.normalize(a_emb, dim=1)
            p_emb = F.normalize(p_emb, dim=1)
            
            val_sim += (a_emb * p_emb).sum(dim=1).sum().item()
            val_count += a_emb.size(0)
    
    val_sim /= val_count
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Sim: {val_sim:.4f}")
    
    if val_sim > best_val_sim:
        best_val_sim = val_sim
        torch.save(model.state_dict(), f"{checkpoint_dir}/best_model.pt")
        print(f"  ✓ Best model saved (val_sim: {best_val_sim:.4f})")

print(f"\n✓ Training complete. Best val sim: {best_val_sim:.4f}")

# 11. Save Final Target Model
final_dir = "../../sparqlce/models"
os.makedirs(final_dir, exist_ok=True)
final_path = f"{final_dir}/dbpedia_model_filter_true_fixed.pt"

torch.save(model.state_dict(), final_path)
print(f"✓ Final model saved to {final_path}")

/upb/users/z/zahera/profiles/unix/cs/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset from ../../data/jaccard_triplets_dbpedia.json...
Loaded 4713 anchor groups.
✓ Training on cuda
✓ Dataset split successfully: Train=3770, Val=471, Test=472


Epoch  1/20 [Train]: 100%|██████████| 118/118 [00:45<00:00,  2.59it/s, ce=1.9940, loss=1.9961, reg=0.0069]


Epoch  1/20 | Train Loss: 2.4306 | Val Sim: 0.9655
  ✓ Best model saved (val_sim: 0.9655)


Epoch  2/20 [Train]: 100%|██████████| 118/118 [00:46<00:00,  2.57it/s, ce=1.3650, loss=1.3677, reg=0.0089]


...

Epoch 18/20 | Train Loss: 0.4283 | Val Sim: 0.8789


Epoch 19/20 [Train]: 100%|██████████| 118/118 [00:46<00:00,  2.56it/s, ce=0.1607, loss=0.1631, reg=0.0079]


Epoch 19/20 | Train Loss: 0.4262 | Val Sim: 0.8812


Epoch 20/20 [Train]: 100%|██████████| 118/118 [00:46<00:00,  2.56it/s, ce=0.3791, loss=0.3835, reg=0.0148]


Epoch 20/20 | Train Loss: 0.3982 | Val Sim: 0.8896

✓ Training complete. Best val sim: 0.9655
✓ Final model saved to ../../sparqlce/models/dbpedia_model_filter_true_fixed.pt
